In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb

In [3]:
df = pd.read_parquet('../data/processed/rossmann_clean.parquet', engine='fastparquet')
print(df.shape)
print(df.dtypes)

(844338, 23)
Store                                 int64
DayOfWeek                             int64
Date                         datetime64[us]
Sales                                 int64
Customers                             int64
Open                                  int64
Promo                                 int64
StateHoliday                       category
SchoolHoliday                         int64
StoreType                          category
Assortment                         category
CompetitionDistance                 float64
CompetitionOpenSinceMonth             int64
CompetitionOpenSinceYear              int64
Promo2                                int64
Promo2SinceWeek                       int64
Promo2SinceYear                       int64
PromoInterval                      category
CompetitionInfoMissing                int64
Year                                  int32
Month                                 int32
Day                                   int32
WeekOfYear         

In [4]:
df['Date'] = pd.to_datetime(df['Date'])
val_start = df['Date'].max() - pd.Timedelta(days=41)
print('Val start: ', val_start)

train = df[df['Date'] < val_start].copy()
val = df[df['Date'] >= val_start].copy()

print('Train: ', train.shape, train['Date'].min().date(), '->', train['Date'].max().date())
print('Val: ', val.shape, val['Date'].min().date(), '->', val['Date'].max().date())
print('Val proportion: ', round((len(val) / len(df)), 4))

Val start:  2015-06-20 00:00:00
Train:  (804056, 23) 2013-01-01 -> 2015-06-19
Val:  (40282, 23) 2015-06-20 -> 2015-07-31
Val proportion:  0.0477


In [5]:
from pandas.api.types import CategoricalDtype

all_stores = sorted(set(train['Store']).union(set(val['Store'])))
store_dtype = CategoricalDtype(categories=all_stores, ordered=False)

train['Store'] = train['Store'].astype(store_dtype)
val['Store'] = val['Store'].astype(store_dtype)

print("n categories train:", train["Store"].cat.categories.size)
print("n categories val:  ", val["Store"].cat.categories.size)
print("categories identical:", (train["Store"].cat.categories == val["Store"].cat.categories).all())
print("stores only in val:", set(val["Store"].unique()) - set(train["Store"].unique()))

n categories train: 1115
n categories val:   1115
categories identical: True
stores only in val: set()


In [6]:
feature_cols = [c for c in train.columns if c not in ['Sales', 'Customers', 'Date', 'Open']]
print('Features: ', len(feature_cols))

def rmspe(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    mask = y_true != 0                      # guard against division by zero
    pct_err = (y_true[mask] - y_pred[mask]) / y_true[mask]
    return np.sqrt(np.mean(pct_err ** 2))

def rmse(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

Features:  19


In [7]:
# Reproducing untuned lightgbm model
train_set = lgb.Dataset(train[feature_cols], label=train["Sales"])
val_set   = lgb.Dataset(val[feature_cols],   label=val["Sales"], reference=train_set)

params = {
    "objective": "regression",   # L2
    "learning_rate": 0.05,
    "random_state": 42,
    "verbose": -1,
}

model = lgb.train(
    params,
    train_set,
    num_boost_round=3000,
    valid_sets=[val_set],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=200)],
)

print("best iteration:", model.best_iteration)

val_pred = model.predict(val[feature_cols], num_iteration=model.best_iteration)
print("RMSPE:", round(rmspe(val["Sales"], val_pred), 4))
print("RMSE: ", round(rmse(val["Sales"], val_pred), 2))

Training until validation scores don't improve for 50 rounds
[200]	valid_0's l2: 1.08462e+06
[400]	valid_0's l2: 951783
[600]	valid_0's l2: 876738
[800]	valid_0's l2: 806836
[1000]	valid_0's l2: 790077
[1200]	valid_0's l2: 771600
[1400]	valid_0's l2: 766727
[1600]	valid_0's l2: 761864
Early stopping, best iteration is:
[1714]	valid_0's l2: 758612
best iteration: 1714
RMSPE: 0.1277
RMSE:  870.98


In [8]:
# cross fold validation
def make_fold(df, val_start, val_days=42):
    val_end = val_start + pd.Timedelta(days=val_days - 1)
    tr = df[df["Date"] < val_start].copy()
    vl = df[(df["Date"] >= val_start) & (df["Date"] <= val_end)].copy()
    # shared Store dtype across this fold's train+val
    stores = sorted(set(tr["Store"]).union(set(vl["Store"])))
    sdt = CategoricalDtype(categories=stores, ordered=False)
    tr["Store"] = tr["Store"].astype(sdt)
    vl["Store"] = vl["Store"].astype(sdt)
    return tr, vl, val_end

def run_fold(tr, vl):
    ts = lgb.Dataset(tr[feature_cols], label=tr["Sales"])
    vs = lgb.Dataset(vl[feature_cols], label=vl["Sales"], reference=ts)
    m = lgb.train(params, ts, num_boost_round=3000, valid_sets=[vs], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    pred = m.predict(vl[feature_cols], num_iteration=m.best_iteration)
    return m.best_iteration, rmspe(vl["Sales"], pred), rmse(vl["Sales"], pred)

# Fold 1 only
f1_start = pd.Timestamp("2015-03-28")
tr1, vl1, f1_end = make_fold(df, f1_start)
print("Fold 1 val:", f1_start.date(), "→", f1_end.date())
print("train rows:", len(tr1), "| val rows:", len(vl1))
print("val-only stores:", set(vl1["Store"].unique()) - set(tr1["Store"].unique()))

it, rp, rm = run_fold(tr1, vl1)
print("best iter:", it, "| RMSPE:", round(rp, 4), "| RMSE:", round(rm, 2))

Fold 1 val: 2015-03-28 → 2015-05-08
train rows: 729540 | val rows: 37044
val-only stores: set()
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[414]	valid_0's l2: 1.42087e+06
best iter: 414 | RMSPE: 0.1601 | RMSE: 1192.0


In [9]:
# Fold 2 only
f2_start = pd.Timestamp("2015-05-09")
tr2, vl2, f2_end = make_fold(df, f2_start)
print("Fold 2 val:", f2_start.date(), "→", f2_end.date())
print("train rows:", len(tr2), "| val rows:", len(vl2))
print("val-only stores:", set(vl2["Store"].unique()) - set(tr2["Store"].unique()))

it, rp, rm = run_fold(tr2, vl2)
print("best iter:", it, "| RMSPE:", round(rp, 4), "| RMSE:", round(rm, 2))

Fold 2 val: 2015-05-09 → 2015-06-19
train rows: 766584 | val rows: 37472
val-only stores: set()
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[520]	valid_0's l2: 873706
best iter: 520 | RMSPE: 0.1335 | RMSE: 934.72


### CV Comparisions
- Fold 1(Mar 28 – May 08): RMSPE-0.1601 RMSE-1192.0 Best iter-414
- Fold 2(May 09 – Jun 19): RMSPE-0.1335 RMSE-934.7 Best iter-520
- Fold 3(headline=Jun 20 – Jul 31): RMSPE-0.1277 RMSE-871.0 Best iter-1714

>The mean CV RMSPE is 0.1404 (average of the three), with a spread of 0.1277 to 0.1601, a range of about 0.032. The honest headline number for this model is therefore ~0.14, not 0.1277. The 0.1277 is the best fold, not the typical fold.

>A 0.032 spread across three windows isn't alarming for an untuned model, and it's explainable (seasonality + training-data volume both trend the same way). So 0.1277 wasn't a fluke in the sense of being wrong; it was real, just unrepresentative. The model is stable enough to trust as a tuning baseline, as long as tuning is done against the CV mean.

In [10]:
# feature importance
imp = pd.DataFrame({
    'feature': model.feature_name(),
    'gain': model.feature_importance(importance_type='gain'),
    'split': model.feature_importance(importance_type='split'),
})

imp['gain_pct'] = (imp['gain'] / imp['gain'].sum() * 100).round(2)
imp = imp.sort_values('gain', ascending=False).reset_index(drop=True)
print(imp.to_string(index=False))

                  feature         gain  split  gain_pct
                    Store 5.423634e+13  21121     71.13
                    Promo 1.101499e+13   2142     14.45
                DayOfWeek 3.848316e+12   5785      5.05
                      Day 2.448905e+12   6127      3.21
               WeekOfYear 2.340622e+12   5488      3.07
                    Month 9.583988e+11   2595      1.26
                     Year 4.563780e+11   2748      0.60
                StoreType 3.043736e+11    429      0.40
      CompetitionDistance 1.837125e+11   1537      0.24
            SchoolHoliday 1.488184e+11    825      0.20
             StateHoliday 8.993530e+10    372      0.12
               Assortment 8.588524e+10    429      0.11
            PromoInterval 3.759278e+10    344      0.05
          Promo2SinceYear 3.371070e+10    269      0.04
          Promo2SinceWeek 2.203743e+10    314      0.03
 CompetitionOpenSinceYear 1.730350e+10    385      0.02
CompetitionOpenSinceMonth 1.431159e+10    410   

> The model is leaning overwhelmingly on store identity and promo, the calendar features fill in seasonality, and roughly half the features are contributing near-zero; several because Store has already absorbed their signal.

In [11]:
v = val.copy()
v['pred'] = model.predict(v[feature_cols], num_iteration=model.best_iteration)
v = v[v['Sales'] != 0].copy()
v['pct_error'] = (v['Sales'] - v['pred']) / v['Sales']
v['abs_pct_error'] = v['pct_error'].abs()

per_store_rmspe = (v.groupby('Store', observed=True).apply(lambda g: np.sqrt(np.mean(g['pct_error'] ** 2))).rename('store_rmspe').reset_index())

print("stores evaluated:", len(per_store_rmspe))
print("median store RMSPE:", round(per_store_rmspe["store_rmspe"].median(), 4))
print("mean store RMSPE:  ", round(per_store_rmspe["store_rmspe"].mean(), 4))
print("\nWorst 15 stores:")
print(per_store_rmspe.sort_values("store_rmspe", ascending=False).head(15).to_string(index=False))
print("\nBest 5 stores:")
print(per_store_rmspe.sort_values("store_rmspe").head(5).to_string(index=False))

stores evaluated: 1115
median store RMSPE: 0.1082
mean store RMSPE:   0.1191

Worst 15 stores:
Store  store_rmspe
  292     1.139725
  909     0.749331
  782     0.537321
  876     0.457993
  550     0.363389
  722     0.325661
  534     0.314993
 1039     0.310569
  575     0.291036
  269     0.283705
  303     0.277724
  612     0.271397
  710     0.267634
  279     0.261015
  169     0.259909

Best 5 stores:
Store  store_rmspe
  817     0.052326
  381     0.055126
  682     0.056718
 1075     0.056740
  539     0.056918


> The best stores hit ~0.05 (half the median). So the achievable floor for a well-behaved store is around 5% error. The spread from 0.05 to 1.14 across stores is enormous, which is itself the finding: store-level predictability varies wildly, and our headline number (0.1277) is an average over very different regimes.

In [12]:
# let's check out store 292 (highest error)
s = df[df['Store'] == 292].sort_values('Date')
print('Total rows: ', len(s))
print('Date Range: ', s['Date'].min().date(), '->', s['Date'].max().date())
print('Rows in panel gap window (2014 h2)?', len(s[(s['Date'] >= '2014-07-01') & (s['Date'] <= '2014-12-31')]))

print("\nSales summary — TRAIN period (before 2015-06-20):")
print(s[s["Date"] < "2015-06-20"]["Sales"].describe().round(1))
print("\nSales summary — VAL period (2015-06-20 onward):")
print(s[s["Date"] >= "2015-06-20"]["Sales"].describe().round(1))

#checking the store's val rows - are they too different from its training rows?
sv = v[v['Store'] == 292][['Date', 'Sales', 'pred', 'pct_error']].sort_values('Date')
print('\nVal rows for store 292:\n')
print(sv.to_string(index=False))

Total rows:  766
Date Range:  2013-01-02 -> 2015-07-10
Rows in panel gap window (2014 h2)? 155

Sales summary — TRAIN period (before 2015-06-20):
count      748.0
mean      5736.7
std       1596.1
min       2367.0
25%       4768.5
50%       5726.5
75%       6624.0
max      10431.0
Name: Sales, dtype: float64

Sales summary — VAL period (2015-06-20 onward):
count       18.0
mean      7328.3
std       7650.8
min       1012.0
25%       3328.2
50%       5045.0
75%       5924.5
max      29161.0
Name: Sales, dtype: float64

Val rows for store 292:

      Date  Sales        pred  pct_error
2015-06-20   3479 3358.992675   0.034495
2015-06-22   6028 5933.386114   0.015696
2015-06-23   5054 5523.470598  -0.092891
2015-06-24   5614 5149.652328   0.082712
2015-06-25   5247 5044.534997   0.038587
2015-06-26   5172 5208.443781  -0.007046
2015-06-27   3278 3345.479670  -0.020586
2015-06-29  29161 8995.995545   0.691506
2015-06-30  23584 8428.731146   0.642608
2015-07-01  15348 7605.082534   0.504490


> The store closed permanently mid-validation. The val rows read: normal sales (~3K–6K) through June 27, matching its training history (mean 5,737). Then June 29–July 1 explode to 29K, 23K, 15K, three days of 3–5× normal. Then a steady collapse: 8K, 5K, 1.6K, down to 1,012 on July 10, after which there are no more rows. Its date range ends 2015-07-10 while every healthy store runs to 2015-07-31. This store stopped trading and the days before closure show a fire-sale spike (the 29K days) followed by wind-down. That's the signature of a store closing: liquidation surge, then decline to zero, then gone.

In [13]:
# finding all such stores that closed on or before 31st July 2015
global_max = df['Date'].max()
print('Global Max Date: ', global_max.date())

last_date = df.groupby('Store', observed=True)['Date'].max().rename('last_date').reset_index()
last_date['day_short'] = (global_max - last_date['last_date']).dt.days

# stores that stop before the val window even ends
stopped = last_date[last_date["last_date"] < global_max].sort_values("last_date")
print("\nstores with last trading date before", global_max.date(), ":", len(stopped))

# of those, how many stop DURING the val window (>= 2015-06-20)?
val_start = pd.Timestamp("2015-06-20")
stop_in_val = stopped[stopped["last_date"] >= val_start]
stop_before_val = stopped[stopped["last_date"] < val_start]
print("  ...stop DURING val window (>= 2015-06-20):", len(stop_in_val))
print("  ...stop BEFORE val window (< 2015-06-20):", len(stop_before_val))

print("\nStores stopping during val window:")
print(stop_in_val.to_string(index=False))

Global Max Date:  2015-07-31

stores with last trading date before 2015-07-31 : 2
  ...stop DURING val window (>= 2015-06-20): 2
  ...stop BEFORE val window (< 2015-06-20): 0

Stores stopping during val window:
 Store  last_date  day_short
   292 2015-07-10         21
   876 2015-07-15         16


> With only 2 stores, the principled exclusion costs almost nothing in sample size and buys a clean metric. 

>Stores that cease trading before the validation window ends are excluded from evaluation, as forecasting sales for a closing store is outside the model's task. Two stores (292, 876) met this criterion.

In [14]:
excluded_stores = [292, 876]

v_clean = v[~v["Store"].isin(excluded_stores)].copy()   # v already has pred + masks zero-sales
print("rows before exclusion:", len(v), "and rows after exclusion:", len(v_clean))
print("stores before:", v["Store"].nunique(), "and stores after:", v_clean["Store"].nunique())

rmspe_clean = np.sqrt(np.mean(v_clean["pct_error"]**2))
rmse_clean  = np.sqrt(np.mean((v_clean["Sales"] - v_clean["pred"])**2))
print("\nfold-3 RMSPE  (contaminated):", 0.1277)
print("fold-3 RMSPE  (cleaned):     ", round(rmspe_clean, 4))
print("fold-3 RMSE   (cleaned):     ", round(rmse_clean, 2))

rows before exclusion: 40282 and rows after exclusion: 40242
stores before: 1115 and stores after: 1113

fold-3 RMSPE  (contaminated): 0.1277
fold-3 RMSPE  (cleaned):      0.125
fold-3 RMSE   (cleaned):      846.18


In [15]:
# similar closure checking for other two folds
def run_fold_clean(df, val_start, val_days=42):
    tr, vl, val_end = make_fold(df, val_start, val_days)
    # exclude stores that stop trading before THIS fold's window ends
    last_in_fold = df.groupby("Store", observed=True)["Date"].max()
    stops = last_in_fold[last_in_fold < val_end].index.tolist()
    vl = vl[~vl["Store"].isin(stops)].copy()
    it, rp, rm = run_fold(tr, vl)
    return it, rp, rm, stops

folds = {
    "Fold 1 (Mar28-May08)": pd.Timestamp("2015-03-28"),
    "Fold 2 (May09-Jun19)": pd.Timestamp("2015-05-09"),
    "Fold 3 (Jun20-Jul31)": pd.Timestamp("2015-06-20"),
}
results = []
for name, start in folds.items():
    it, rp, rm, stops = run_fold_clean(df, start)
    results.append((name, rp, rm, it, stops))
    print(f"{name}: RMSPE {rp:.4f} | RMSE {rm:.2f} | iter {it} | excluded {stops}")

mean_rmspe = np.mean([r[1] for r in results])
print(f"\nCV mean RMSPE (cleaned): {mean_rmspe:.4f}")

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[414]	valid_0's l2: 1.42087e+06
Fold 1 (Mar28-May08): RMSPE 0.1601 | RMSE 1192.00 | iter 414 | excluded []
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[520]	valid_0's l2: 873706
Fold 2 (May09-Jun19): RMSPE 0.1335 | RMSE 934.72 | iter 520 | excluded []
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1714]	valid_0's l2: 716022
Fold 3 (Jun20-Jul31): RMSPE 0.1250 | RMSE 846.18 | iter 1714 | excluded [292, 876]

CV mean RMSPE (cleaned): 0.1395


### Evaluation Phase Key Points
- CV mean RMSPE 0.1395 (range 0.125–0.160), with the spread explained by seasonality + training-volume both trending the same way.

- Error is concentrated, not uniform; median store 0.108, a right-skewed tail, two stores were closing-store artifacts (now excluded on a documented rule).

- Feature reality: Store (71%) and Promo (14%) dominate; calendar features fill seasonality; StateHoliday earns ~0.12% (kept honestly, not for signal); CompetitionInfoMissing is inert (3 splits).

- Unstable tree count across folds (414/520/1714): the standing warning that early-stopping on a single window is not a stable way to set tree count.

In [16]:
# log transforming sales to resolve objective mismatch (l2 and rmspe)
# fold 3 train/val and Store already shared-dtype
y_train_log = np.log1p(train["Sales"])
y_val_log   = np.log1p(val["Sales"])

t_log = lgb.Dataset(train[feature_cols], label=y_train_log)
v_log = lgb.Dataset(val[feature_cols],   label=y_val_log, reference=t_log)

model_log = lgb.train(
    params, t_log, num_boost_round=3000, valid_sets=[v_log], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
)
print("best iteration:", model_log.best_iteration)

# predict in log space, back-transform to real sales
val_pred_log = model_log.predict(val[feature_cols], num_iteration=model_log.best_iteration)
val_pred = np.expm1(val_pred_log)           # inverse of log1p — back to real units

# measure on val (excluding 292, 876) to compare with 0.125
mask_clean = ~val["Store"].isin([292, 876]) & (val["Sales"] != 0)
yt = val["Sales"][mask_clean].values
yp = val_pred[mask_clean]

print("\nL2-trained fold-3 RMSPE (cleaned):  0.1250  (prior)")
print("log-trained fold-3 RMSPE (cleaned):", round(rmspe(yt, yp), 4))
print("log-trained fold-3 RMSE  (cleaned):", round(rmse(yt, yp), 2))

# back-transform bias check: mean actual vs mean predicted across cleaned val
print("\nmean actual:", round(yt.mean(), 1), "| mean pred:", round(yp.mean(), 1), "| pred/actual:", round(yp.mean()/yt.mean(), 4))

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1542]	valid_0's l2: 0.0142155
best iteration: 1542

L2-trained fold-3 RMSPE (cleaned):  0.1250  (prior)
log-trained fold-3 RMSPE (cleaned): 0.1272
log-trained fold-3 RMSE  (cleaned): 870.85

mean actual: 6976.1 | mean pred: 7027.9 | pred/actual: 1.0074


In [17]:
# running log-transform on all CV folds
def run_fold_log(df, val_start, val_days=42):
    tr, vl, val_end = make_fold(df, val_start, val_days)
    last_in_fold = df.groupby("Store", observed=True)["Date"].max()
    stops = last_in_fold[last_in_fold < val_end].index.tolist()
    vl = vl[~vl["Store"].isin(stops)].copy()

    ts = lgb.Dataset(tr[feature_cols], label=np.log1p(tr["Sales"]))
    vs = lgb.Dataset(vl[feature_cols], label=np.log1p(vl["Sales"]), reference=ts)
    m = lgb.train(params, ts, num_boost_round=3000, valid_sets=[vs],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    pred = np.expm1(m.predict(vl[feature_cols], num_iteration=m.best_iteration))
    return m.best_iteration, rmspe(vl["Sales"], pred), rmse(vl["Sales"], pred), stops

print(f"{'fold':<22}{'log RMSPE':<12}{'L2 RMSPE':<12}{'iter':<8}")
prior = {"Fold 1 (Mar28-May08)": 0.1601, "Fold 2 (May09-Jun19)": 0.1335, "Fold 3 (Jun20-Jul31)": 0.1250}
log_rmspes = []
for name, start in folds.items():
    it, rp, rm, stops = run_fold_log(df, start)
    log_rmspes.append(rp)
    print(f"{name:<22}{rp:<12.4f}{prior[name]:<12.4f}{it:<8}")

print(f"\nlog CV mean: {np.mean(log_rmspes):.4f}  vs  L2 CV mean: 0.1395")

fold                  log RMSPE   L2 RMSPE    iter    
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[359]	valid_0's l2: 0.0210086
Fold 1 (Mar28-May08)  0.1422      0.1601      359     
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[544]	valid_0's l2: 0.0166713
Fold 2 (May09-Jun19)  0.1256      0.1335      544     
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1542]	valid_0's l2: 0.0138654
Fold 3 (Jun20-Jul31)  0.1272      0.1250      1542    

log CV mean: 0.1317  vs  L2 CV mean: 0.1395


### This is why we tested across folds instead of trusting fold 3.
Log CV mean 0.1317 vs L2 0.1395: log wins. That's a ~5.6% relative improvement in the honest headline number.

Fold 1 — the hard window where log's small-store advantage should show up, improved the most. Fold 3, the big-store window, got marginally worse. This is the theory playing out: log-transform helps where percentage error diverges from absolute error (small stores, volatile windows) and is roughly neutral where it doesn't. Fold 3 alone misled us because it's the one window where log's benefit is smallest.

The log-transform does help here, but its benefit is concentrated in the harder-to-predict windows.

**Updated headline:** CV mean RMSPE 0.1317 (range 0.1272–0.1422), log-trained, closing stores excluded per-fold. Note the range tightened too (0.015 spread vs 0.032 before). Log-transform didn't just lower the mean, it made the model more consistent across windows, because it stopped the hard windows from blowing out.

From here, the remaining tuning builds on the log-trained model and gets measured against 0.1317

In [18]:
# high-cardinality store handling (1115 unique stores)
params_cat = dict(params)
params_cat.update({
    "max_cat_threshold": 128,     # up from default 32 ; finer store groupings
    "min_data_per_group": 50,     # down from default 100 ; allow smaller store groups
    "cat_smooth": 10,             # default
})

def run_fold_log_params(df, val_start, p, val_days=42):
    tr, vl, val_end = make_fold(df, val_start, val_days)
    last_in_fold = df.groupby("Store", observed=True)["Date"].max()
    stops = last_in_fold[last_in_fold < val_end].index.tolist()
    vl = vl[~vl["Store"].isin(stops)].copy()
    ts = lgb.Dataset(tr[feature_cols], label=np.log1p(tr["Sales"]))
    vs = lgb.Dataset(vl[feature_cols], label=np.log1p(vl["Sales"]), reference=ts)
    m = lgb.train(p, ts, num_boost_round=3000, valid_sets=[vs],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    pred = np.expm1(m.predict(vl[feature_cols], num_iteration=m.best_iteration))
    return m.best_iteration, rmspe(vl["Sales"], pred)

base_log = {"Fold 1 (Mar28-May08)": 0.1422, "Fold 2 (May09-Jun19)": 0.1256, "Fold 3 (Jun20-Jul31)": 0.1272}
cat_rmspes = []
print(f"{'fold':<22}{'cat-tuned':<12}{'log base':<12}{'iter':<8}")
for name, start in folds.items():
    it, rp = run_fold_log_params(df, start, params_cat)
    cat_rmspes.append(rp)
    print(f"{name:<22}{rp:<12.4f}{base_log[name]:<12.4f}{it:<8}")
print(f"\ncat-tuned CV mean: {np.mean(cat_rmspes):.4f}  vs  log base: 0.1317")

fold                  cat-tuned   log base    iter    
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[462]	valid_0's l2: 0.0197885
Fold 1 (Mar28-May08)  0.1381      0.1422      462     
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[463]	valid_0's l2: 0.0158738
Fold 2 (May09-Jun19)  0.1235      0.1256      463     
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[760]	valid_0's l2: 0.0139244
Fold 3 (Jun20-Jul31)  0.1274      0.1272      760     

cat-tuned CV mean: 0.1297  vs  log base: 0.1317


**Updated headline:** CV mean RMSPE 0.1297 (range 0.1235–0.1381), log-trained + cat-tuned, closing stores excluded per-fold.

In [19]:
#hp tuning - learning rate lowering
def cv_eval(df, p, label):
    rmspes, iters = [], []
    for name, start in folds.items():
        it, rp = run_fold_log_params(df, start, p)
        rmspes.append(rp)
        iters.append(it)

    print(f"{label:<18} CV mean {np.mean(rmspes):.4f} | folds "
          f"[{rmspes[0]:.4f}, {rmspes[1]:.4f}, {rmspes[2]:.4f}] | iters {iters}")
    return np.mean(rmspes)

for lr in [0.05, 0.03, 0.02]:
    p = dict(params_cat); p["learning_rate"] = lr
    cv_eval(df, p, f"lr={lr}")

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[462]	valid_0's l2: 0.0197885
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[463]	valid_0's l2: 0.0158738
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[760]	valid_0's l2: 0.0139244
lr=0.05            CV mean 0.1297 | folds [0.1381, 0.1235, 0.1274] | iters [462, 463, 760]
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[681]	valid_0's l2: 0.020055
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[684]	valid_0's l2: 0.0158745
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1794]	valid_0's l2: 0.0128628
lr=0.03            CV mean 0.1276 | folds [0.1382, 0.1232, 0.1213] | iters [681, 684, 1794]
Training until validation scores don't improve for 50 rounds


> The entire gain came from fold 3 (0.1274 → 0.1213). Folds 1 and 2 are essentially flat across all three rates. This breaks the pattern from the last two interventions, log and cat-tuning helped the hard folds (1&2); learning rate helped the easy fold 3. That actually makes sense: fold 3 has the most training data and the most trees, so it has the most to gain from careful convergence (low learning rate lets it exploit that data fully). The hard folds are limited by signal, so a gentler step size doesn't help them.

> Tree count remains window-dependent, that's intrinsic to the data, not something learning rate fixes.

> Locking in lr=0.03. Updated headline: CV mean 0.1276 (range 0.1213–0.1382), log + cat-tuned + lr=0.03.

In [20]:
for num_leaves in [15, 31, 63, 127]:
    p = dict(params_cat)
    p['learning_rate'] = 0.03
    p['num_leaves'] = num_leaves
    cv_eval(df, p, f'num_leaves={num_leaves}')

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[922]	valid_0's l2: 0.0190144
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1655]	valid_0's l2: 0.0150934
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2712]	valid_0's l2: 0.0133715
num_leaves=15      CV mean 0.1264 | folds [0.1342, 0.1207, 0.1244] | iters [922, 1655, 2712]
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[681]	valid_0's l2: 0.020055
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[684]	valid_0's l2: 0.0158745
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1794]	valid_0's l2: 0.0128628
num_leaves=31      CV mean 0.1276 | folds [0.1382, 0.1232, 0.1213] | iters [681, 684, 1794]
Training until validation scores don't improve for 50 rou

> num_leaves=15 helps folds 1 and 2 but slightly hurts fold 3 (0.1213 → 0.1244 vs the 31-leaf result). Fold 3 has the most data, so it's the one window that can productively use slightly more complexity. So 15 isn't a free win; it trades a little fold-3 accuracy for larger gains on the harder folds. But since folds 1 and 2 are where the error is highest and where real-world hard cases live, improving them at fold 3's minor expense is the right trade and the CV mean agrees (0.1264 < 0.1276).

In [21]:
# min-child-sample
basep = dict(params_cat)
basep.update({'learning_rate': 0.03, 'num_leaves': 15})

for mcs in [20, 50, 100, 200]:
    p = dict(basep)
    p['min_child_samples'] = mcs
    cv_eval(df, p, f'min_child_samples={mcs}')

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[922]	valid_0's l2: 0.0190144
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1655]	valid_0's l2: 0.0150934
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2712]	valid_0's l2: 0.0133715
min_child_samples=20 CV mean 0.1264 | folds [0.1342, 0.1207, 0.1244] | iters [922, 1655, 2712]
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[970]	valid_0's l2: 0.0194449
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1174]	valid_0's l2: 0.0155565
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2024]	valid_0's l2: 0.0134423
min_child_samples=50 CV mean 0.1275 | folds [0.1354, 0.1228, 0.1242] | iters [970, 1174, 2024]
Training until validation scores don't improve for

> We keep the min_child_sample parameter on its default (20). Every increase made it worse or neutral (50 worse, 100 worst, 200 recovers slightly but still doesn't beat 20). There's no trend pointing anywhere useful.

>The regularizer confirms there's no residual overfitting left for us to handle. 

### We've hit diminishing returns on tuning. Look at the arc:
- log-transform: −0.008 (0.1395 → 0.1317)
- cat-tuning: −0.002 (0.1317 → 0.1297)
- lr=0.03: −0.002 (0.1297 → 0.1276)
- num_leaves=15: −0.001 (0.1276 → 0.1264)
- min_child_samples: 0.000 (no improvement)

In [23]:
# retrain fold-3 (fold with most training data and latest ending) cleanly with the final config
final_p = dict(params_cat)
final_p.update({'learning_rate': 0.03, 'num_leaves': 15})
print('Final Config: ', {k: final_p[k] for k in ['objective', 'learning_rate', 'num_leaves', "max_cat_threshold", "min_data_per_group", "cat_smooth", "random_state"]})

# train final model on fold-3 split, log target
t = lgb.Dataset(train[feature_cols], label=np.log1p(train["Sales"]))
v = lgb.Dataset(val[feature_cols],   label=np.log1p(val["Sales"]), reference=t)
final_model = lgb.train(final_p, t, num_boost_round=3000, valid_sets=[v], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
print("best iteration:", final_model.best_iteration)

# predict, back-transform, evaluate on CLEANED val (exclude 292, 876)
pred = np.expm1(final_model.predict(val[feature_cols], num_iteration=final_model.best_iteration))
m = ~val["Store"].isin([292, 876]) & (val["Sales"] != 0)
print("final fold-3 RMSPE (cleaned):", round(rmspe(val["Sales"][m], pred[m]), 4))
print("final fold-3 RMSE  (cleaned):", round(rmse(val["Sales"][m], pred[m]), 2))

# save the model
final_model.save_model("../outputs/models/lgbm_final.txt")
print("saved -> outputs/models/lgbm_final.txt")

Final Config:  {'objective': 'regression', 'learning_rate': 0.03, 'num_leaves': 15, 'max_cat_threshold': 128, 'min_data_per_group': 50, 'cat_smooth': 10, 'random_state': 42}
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2712]	valid_0's l2: 0.013723
best iteration: 2712
final fold-3 RMSPE (cleaned): 0.1244
final fold-3 RMSE  (cleaned): 853.77
saved -> outputs/models/lgbm_final.txt
